### Import required libraries

In [1]:
import os
import time
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 4090


### Load the data from the files

In [5]:
from pathlib import Path
from data_preprocessing import create_dataloader

# Dataset paths
SPLIT_ROOT = Path(
    "/app/DATA266-Lab1-Pair45/task3_GAN/dataset/split_data"
)

PHOTO_TRAIN_DIR = SPLIT_ROOT / "train" / "photos"
MONET_TRAIN_DIR = SPLIT_ROOT / "train" / "monet"

print("Photo train folder:", PHOTO_TRAIN_DIR)
print("Monet train folder:", MONET_TRAIN_DIR)

print("Photo folder exists:", PHOTO_TRAIN_DIR.exists())
print("Monet folder exists:", MONET_TRAIN_DIR.exists())


# Create the training DataLoader
train_loader = create_dataloader(
    domain_a_dir=PHOTO_TRAIN_DIR,
    domain_b_dir=MONET_TRAIN_DIR,
    batch_size=16,
    train=True,
    shuffle=True,
    num_workers=0
)

print("Number of batches:", len(train_loader))

Photo train folder: /app/DATA266-Lab1-Pair45/task3_GAN/dataset/split_data/train/photos
Monet train folder: /app/DATA266-Lab1-Pair45/task3_GAN/dataset/split_data/train/monet
Photo folder exists: True
Monet folder exists: True
Number of batches: 352


### Create the networks

In [6]:
from models_v2 import (
    ImprovedGenerator,
    LargePatchGANDiscriminator,
    count_parameters
)

# Create two generators
G_AB = ImprovedGenerator(
    input_channels=3,
    output_channels=3,
    num_residual_blocks=9
).to(device)

G_BA = ImprovedGenerator(
    input_channels=3,
    output_channels=3,
    num_residual_blocks=9
).to(device)

# Create two larger-receptive-field discriminators
D_A = LargePatchGANDiscriminator(
    input_channels=3
).to(device)

D_B = LargePatchGANDiscriminator(
    input_channels=3
).to(device)

# Set training mode
G_AB.train()
G_BA.train()
D_A.train()
D_B.train()

print("Networks created successfully.")
print("G_AB parameters:", count_parameters(G_AB))
print("G_BA parameters:", count_parameters(G_BA))
print("D_A parameters:", count_parameters(D_A))
print("D_B parameters:", count_parameters(D_B))

Networks created successfully.
G_AB parameters: 11372931
G_BA parameters: 11372931
D_A parameters: 6959553
D_B parameters: 6959553


In [7]:
batch = next(iter(train_loader))

print("Batch loaded.")
print("Photo shape:", batch["A"].shape)
print("Monet shape:", batch["B"].shape)

Batch loaded.
Photo shape: torch.Size([16, 3, 256, 256])
Monet shape: torch.Size([16, 3, 256, 256])


In [8]:
real_A = batch["A"].to(device)
real_B = batch["B"].to(device)

with torch.no_grad():
    fake_B = G_AB(real_A)
    fake_A = G_BA(real_B)

    prediction_A = D_A(fake_A)
    prediction_B = D_B(fake_B)

print("Real A shape:", real_A.shape)
print("Fake B shape:", fake_B.shape)
print("Real B shape:", real_B.shape)
print("Fake A shape:", fake_A.shape)

print("Discriminator A output:", prediction_A.shape)
print("Discriminator B output:", prediction_B.shape)

Real A shape: torch.Size([16, 3, 256, 256])
Fake B shape: torch.Size([16, 3, 256, 256])
Real B shape: torch.Size([16, 3, 256, 256])
Fake A shape: torch.Size([16, 3, 256, 256])
Discriminator A output: torch.Size([16, 1, 14, 14])
Discriminator B output: torch.Size([16, 1, 14, 14])


### Define the losses

In [9]:
import torch.nn as nn

# Least-squares adversarial loss used by CycleGAN
criterion_gan = nn.MSELoss()

# Cycle-consistency loss
criterion_cycle = nn.L1Loss()

# Identity-mapping loss
criterion_identity = nn.L1Loss()

# Loss weights
lambda_cycle = 10.0
lambda_identity = 5.0

print("Loss functions created.")
print("lambda_cycle:", lambda_cycle)
print("lambda_identity:", lambda_identity)

Loss functions created.
lambda_cycle: 10.0
lambda_identity: 5.0


### Create the optimizers

In [10]:
import itertools
import torch.optim as optim

generator_lr = 0.0002
discriminator_lr = 0.0001

beta1 = 0.5
beta2 = 0.999

optimizer_G = optim.Adam(
    itertools.chain(
        G_AB.parameters(),
        G_BA.parameters()
    ),
    lr=generator_lr,
    betas=(beta1, beta2)
)

optimizer_D_A = optim.Adam(
    D_A.parameters(),
    lr=discriminator_lr,
    betas=(beta1, beta2)
)

optimizer_D_B = optim.Adam(
    D_B.parameters(),
    lr=discriminator_lr,
    betas=(beta1, beta2)
)

print("Optimizers created successfully.")

Optimizers created successfully.


In [11]:
from torch.optim.lr_scheduler import LambdaLR

NUM_EPOCHS = 100
DECAY_START_EPOCH = 50


def linear_decay(epoch):

    # Keep the learning rate constant during the first half
    if epoch < DECAY_START_EPOCH:
        return 1.0

    # Linearly reduce the learning rate during the second half
    decay_epochs = NUM_EPOCHS - DECAY_START_EPOCH

    return max(
        0.0,
        1.0 - (
            epoch - DECAY_START_EPOCH
        ) / decay_epochs
    )


scheduler_G = LambdaLR(
    optimizer_G,
    lr_lambda=linear_decay
)

scheduler_D_A = LambdaLR(
    optimizer_D_A,
    lr_lambda=linear_decay
)

scheduler_D_B = LambdaLR(
    optimizer_D_B,
    lr_lambda=linear_decay
)

print("Learning-rate schedulers created.")
print("Total epochs:", NUM_EPOCHS)
print("Learning-rate decay starts at epoch:", DECAY_START_EPOCH)

Learning-rate schedulers created.
Total epochs: 100
Learning-rate decay starts at epoch: 50


### Add fake-image replay buffers

In [12]:
import random
import torch


class ImagePool:

    def __init__(self, pool_size=50):
        self.pool_size = pool_size
        self.images = []
        self.num_images = 0

    def query(self, images):

        if self.pool_size == 0:
            return images

        returned_images = []

        for image in images:

            image = image.unsqueeze(0)

            # Fill the pool initially
            if self.num_images < self.pool_size:

                self.images.append(image[0].detach().clone())
                self.num_images += 1

                returned_images.append(image)

            else:

                # 50% chance of using an older generated image
                if random.random() > 0.5:

                    random_index = random.randint(
                        0,
                        self.pool_size - 1
                    )

                    old_image = (
                        self.images[random_index]
                        .unsqueeze(0)
                        .clone()
                    )

                    self.images[random_index] = (
                        image[0].detach().clone()
                    )

                    returned_images.append(old_image)

                else:

                    returned_images.append(image)

        return torch.cat(returned_images, dim=0)


# One pool for each translation direction
fake_pool_A = ImagePool(pool_size=50)
fake_pool_B = ImagePool(pool_size=50)

print("Fake-image replay buffers created.")
print("Pool size:", 50)

Fake-image replay buffers created.
Pool size: 50


### Prepare training configuration

In [13]:
from pathlib import Path
import time

# Training configuration
NUM_EPOCHS = 100
PRINT_EVERY = 1

# Checkpoint folder for this new experiment
CHECKPOINT_DIR = Path("checkpoints_v3")
CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Automatic mixed precision
use_amp = torch.cuda.is_available()

scaler = torch.cuda.amp.GradScaler(
    enabled=use_amp
)

# Store average losses for plotting later
history = {
    "generator": [],
    "discriminator_A": [],
    "discriminator_B": [],
    "cycle": [],
    "identity": [],
    "learning_rate_G": [],
    "learning_rate_D": []
}

print("Training configuration ready.")
print("Total epochs:", NUM_EPOCHS)
print("AMP enabled:", use_amp)
print("Checkpoint folder:", CHECKPOINT_DIR.resolve())

Training configuration ready.
Total epochs: 100
AMP enabled: True
Checkpoint folder: /app/DATA266-Lab1-Pair45/task3_GAN/Samruddhi/src/checkpoints_v3


### Training

In [ ]:
# ============================================================
# CycleGAN Training Loop
# ============================================================

import os
import time
from pathlib import Path

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def set_requires_grad(models, requires_grad):
    for model in models:
        for parameter in model.parameters():
            parameter.requires_grad = requires_grad


def safe_torch_save(checkpoint, path):
    """
    Save to a temporary file first, then replace the final file.
    This prevents incomplete checkpoint files.
    """
    path = Path(path)
    temporary_path = Path(str(path) + ".tmp")

    torch.save(checkpoint, temporary_path)
    os.replace(temporary_path, path)


def current_learning_rate(optimizer):
    return optimizer.param_groups[0]["lr"]


# ------------------------------------------------------------
# Training settings
# ------------------------------------------------------------

NUM_EPOCHS = 100
PRINT_EVERY = 10

CHECKPOINT_DIR = Path("checkpoints_v3")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

best_monitor_score = float("inf")
training_start_time = time.time()


# ------------------------------------------------------------
# Reset history in case this cell is run again
# ------------------------------------------------------------

history = {
    "generator": [],
    "discriminator_A": [],
    "discriminator_B": [],
    "cycle": [],
    "identity": [],
    "learning_rate_G": [],
    "learning_rate_D": []
}


# ------------------------------------------------------------
# Start training
# ------------------------------------------------------------

print("=" * 80)
print("Training started.")
print("Total epochs:", NUM_EPOCHS)
print("Batches per epoch:", len(train_loader))
print("Device:", device)
print("AMP enabled:", use_amp)
print("Checkpoint directory:", CHECKPOINT_DIR.resolve())
print("=" * 80, flush=True)


for epoch in range(NUM_EPOCHS):

    epoch_start_time = time.time()

    # Put all models in training mode
    G_AB.train()
    G_BA.train()
    D_A.train()
    D_B.train()

    # Running loss totals
    total_loss_G = 0.0
    total_loss_D_A = 0.0
    total_loss_D_B = 0.0
    total_loss_cycle = 0.0
    total_loss_identity = 0.0

    number_of_batches = 0

    print(
        f"\nStarting Epoch [{epoch + 1}/{NUM_EPOCHS}]...",
        flush=True
    )


    # ========================================================
    # Process every batch
    # ========================================================

    for batch_index, batch in enumerate(train_loader, start=1):

        real_A = batch["A"].to(device, non_blocking=True)
        real_B = batch["B"].to(device, non_blocking=True)

        # ----------------------------------------------------
        # Step 1: Train both generators
        # ----------------------------------------------------

        # Freeze discriminators
        set_requires_grad([D_A, D_B], False)

        optimizer_G.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):

            # A = Photos
            # B = Monet paintings
            #
            # G_AB: Photos -> Monet
            # G_BA: Monet -> Photos

            fake_B = G_AB(real_A)
            fake_A = G_BA(real_B)

            # Cycle reconstruction
            reconstructed_A = G_BA(fake_B)
            reconstructed_B = G_AB(fake_A)

            # Identity mapping
            identity_A = G_BA(real_A)
            identity_B = G_AB(real_B)

            # Discriminator predictions for generated images
            prediction_fake_B = D_B(fake_B)
            prediction_fake_A = D_A(fake_A)

            # Generator adversarial losses
            loss_G_AB = criterion_gan(
                prediction_fake_B,
                torch.ones_like(prediction_fake_B)
            )

            loss_G_BA = criterion_gan(
                prediction_fake_A,
                torch.ones_like(prediction_fake_A)
            )

            # Cycle-consistency loss
            loss_cycle_A = criterion_cycle(
                reconstructed_A,
                real_A
            )

            loss_cycle_B = criterion_cycle(
                reconstructed_B,
                real_B
            )

            loss_cycle_total = loss_cycle_A + loss_cycle_B

            # Identity loss
            loss_identity_A = criterion_identity(
                identity_A,
                real_A
            )

            loss_identity_B = criterion_identity(
                identity_B,
                real_B
            )

            loss_identity_total = (
                loss_identity_A + loss_identity_B
            )

            # Total generator loss
            loss_G = (
                loss_G_AB
                + loss_G_BA
                + lambda_cycle * loss_cycle_total
                + lambda_identity * loss_identity_total
            )

        # Update generators
        scaler.scale(loss_G).backward()
        scaler.step(optimizer_G)


        # ----------------------------------------------------
        # Step 2: Train discriminator D_A
        # ----------------------------------------------------

        set_requires_grad([D_A, D_B], True)

        optimizer_D_A.zero_grad(set_to_none=True)

        # Use replay buffer for generated A images
        fake_A_for_discriminator = fake_pool_A.query(
            fake_A.detach()
        )

        with torch.cuda.amp.autocast(enabled=use_amp):

            prediction_real_A = D_A(real_A)
            prediction_fake_A_pool = D_A(
                fake_A_for_discriminator
            )

            loss_D_A_real = criterion_gan(
                prediction_real_A,
                torch.ones_like(prediction_real_A)
            )

            loss_D_A_fake = criterion_gan(
                prediction_fake_A_pool,
                torch.zeros_like(prediction_fake_A_pool)
            )

            loss_D_A = 0.5 * (
                loss_D_A_real + loss_D_A_fake
            )

        scaler.scale(loss_D_A).backward()
        scaler.step(optimizer_D_A)


        # ----------------------------------------------------
        # Step 3: Train discriminator D_B
        # ----------------------------------------------------

        optimizer_D_B.zero_grad(set_to_none=True)

        # Use replay buffer for generated B images
        fake_B_for_discriminator = fake_pool_B.query(
            fake_B.detach()
        )

        with torch.cuda.amp.autocast(enabled=use_amp):

            prediction_real_B = D_B(real_B)
            prediction_fake_B_pool = D_B(
                fake_B_for_discriminator
            )

            loss_D_B_real = criterion_gan(
                prediction_real_B,
                torch.ones_like(prediction_real_B)
            )

            loss_D_B_fake = criterion_gan(
                prediction_fake_B_pool,
                torch.zeros_like(prediction_fake_B_pool)
            )

            loss_D_B = 0.5 * (
                loss_D_B_real + loss_D_B_fake
            )

        scaler.scale(loss_D_B).backward()
        scaler.step(optimizer_D_B)

        # Update AMP scaler after all optimizer steps
        scaler.update()


        # ----------------------------------------------------
        # Check for invalid losses
        # ----------------------------------------------------

        losses_are_valid = all(
            np.isfinite(value)
            for value in [
                loss_G.item(),
                loss_D_A.item(),
                loss_D_B.item(),
                loss_cycle_total.item(),
                loss_identity_total.item()
            ]
        )

        if not losses_are_valid:
            print(
                f"WARNING: Invalid loss detected at "
                f"epoch {epoch + 1}, batch {batch_index}",
                flush=True
            )
            raise RuntimeError(
                "Training stopped because a NaN or infinity was detected."
            )


        # ----------------------------------------------------
        # Store batch losses
        # ----------------------------------------------------

        total_loss_G += loss_G.item()
        total_loss_D_A += loss_D_A.item()
        total_loss_D_B += loss_D_B.item()
        total_loss_cycle += loss_cycle_total.item()
        total_loss_identity += loss_identity_total.item()

        number_of_batches += 1


        # ----------------------------------------------------
        # Verbose batch output
        # ----------------------------------------------------

        if (
            batch_index == 1
            or batch_index % PRINT_EVERY == 0
            or batch_index == len(train_loader)
        ):
            print(
                f"Epoch [{epoch + 1}/{NUM_EPOCHS}] | "
                f"Batch [{batch_index}/{len(train_loader)}] | "
                f"G: {loss_G.item():.4f} | "
                f"D_A: {loss_D_A.item():.4f} | "
                f"D_B: {loss_D_B.item():.4f} | "
                f"Cycle: {loss_cycle_total.item():.4f} | "
                f"Identity: {loss_identity_total.item():.4f}",
                flush=True
            )


    # ========================================================
    # Calculate average epoch losses
    # ========================================================

    average_loss_G = total_loss_G / number_of_batches
    average_loss_D_A = total_loss_D_A / number_of_batches
    average_loss_D_B = total_loss_D_B / number_of_batches
    average_loss_cycle = total_loss_cycle / number_of_batches
    average_loss_identity = total_loss_identity / number_of_batches

    # Update learning-rate schedulers after the epoch
    scheduler_G.step()
    scheduler_D_A.step()
    scheduler_D_B.step()

    learning_rate_G = current_learning_rate(optimizer_G)
    learning_rate_D = current_learning_rate(optimizer_D_A)

    # Store history
    history["generator"].append(average_loss_G)
    history["discriminator_A"].append(average_loss_D_A)
    history["discriminator_B"].append(average_loss_D_B)
    history["cycle"].append(average_loss_cycle)
    history["identity"].append(average_loss_identity)
    history["learning_rate_G"].append(learning_rate_G)
    history["learning_rate_D"].append(learning_rate_D)


    # ========================================================
    # Epoch summary
    # ========================================================

    epoch_time = time.time() - epoch_start_time

    print("\n" + "-" * 80)
    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] completed "
        f"in {epoch_time / 60:.2f} minutes"
    )
    print(f"Average Generator loss:       {average_loss_G:.6f}")
    print(f"Average Discriminator A loss: {average_loss_D_A:.6f}")
    print(f"Average Discriminator B loss: {average_loss_D_B:.6f}")
    print(f"Average Cycle loss:            {average_loss_cycle:.6f}")
    print(f"Average Identity loss:         {average_loss_identity:.6f}")
    print(f"Generator learning rate:       {learning_rate_G:.8f}")
    print(f"Discriminator learning rate:   {learning_rate_D:.8f}")
    print("-" * 80, flush=True)


    # ========================================================
    # Save latest checkpoint
    # ========================================================

    monitor_score = (
        average_loss_cycle
        + average_loss_identity
    )

    checkpoint = {
        "epoch": epoch + 1,

        "G_AB": G_AB.state_dict(),
        "G_BA": G_BA.state_dict(),
        "D_A": D_A.state_dict(),
        "D_B": D_B.state_dict(),

        "optimizer_G": optimizer_G.state_dict(),
        "optimizer_D_A": optimizer_D_A.state_dict(),
        "optimizer_D_B": optimizer_D_B.state_dict(),

        "scheduler_G": scheduler_G.state_dict(),
        "scheduler_D_A": scheduler_D_A.state_dict(),
        "scheduler_D_B": scheduler_D_B.state_dict(),

        "scaler": scaler.state_dict(),

        "history": history,

        "monitor_score": monitor_score,

        "config": {
            "generator_lr": generator_lr,
            "discriminator_lr": discriminator_lr,
            "lambda_cycle": lambda_cycle,
            "lambda_identity": lambda_identity,
            "batch_size": train_loader.batch_size,
            "num_epochs": NUM_EPOCHS,
            "decay_start_epoch": DECAY_START_EPOCH,
            "amp_enabled": use_amp
        }
    }

    latest_path = CHECKPOINT_DIR / "latest_cyclegan_v3.pth"

    safe_torch_save(
        checkpoint,
        latest_path
    )

    print(
        "Latest checkpoint saved:",
        latest_path.resolve(),
        flush=True
    )


    # ========================================================
    # Save best checkpoint
    # ========================================================

    if monitor_score < best_monitor_score:

        best_monitor_score = monitor_score

        checkpoint["best_monitor_score"] = best_monitor_score

        best_path = CHECKPOINT_DIR / "best_cyclegan_v3.pth"

        safe_torch_save(
            checkpoint,
            best_path
        )

        print(
            "New best checkpoint saved:",
            best_path.resolve(),
            flush=True
        )

    else:

        print(
            "No improvement in cycle + identity monitor.",
            flush=True
        )


# ============================================================
# Training completed
# ============================================================

total_training_time = time.time() - training_start_time

print("\n" + "=" * 80)
print("Training completed.")
print(
    f"Total training time: "
    f"{total_training_time / 60:.2f} minutes"
)
print(
    "Best checkpoint:",
    (CHECKPOINT_DIR / "best_cyclegan_v3.pth").resolve()
)
print(
    "Latest checkpoint:",
    (CHECKPOINT_DIR / "latest_cyclegan_v3.pth").resolve()
)
print("=" * 80)

Training started.
Total epochs: 100
Batches per epoch: 352
Device: cuda
AMP enabled: True
Checkpoint directory: /app/DATA266-Lab1-Pair45/task3_GAN/Samruddhi/src/checkpoints_v3

Starting Epoch [1/100]...
Epoch [1/100] | Batch [1/352] | G: 21.0987 | D_A: 1.9123 | D_B: 1.2384 | Cycle: 1.0640 | Identity: 1.0777
Epoch [1/100] | Batch [10/352] | G: 11.1465 | D_A: 0.5362 | D_B: 0.5029 | Cycle: 0.6830 | Identity: 0.6518
Epoch [1/100] | Batch [20/352] | G: 9.0165 | D_A: 0.3865 | D_B: 0.4655 | Cycle: 0.5538 | Identity: 0.5211
Epoch [1/100] | Batch [30/352] | G: 10.4325 | D_A: 0.3339 | D_B: 0.2897 | Cycle: 0.6747 | Identity: 0.5727
Epoch [1/100] | Batch [40/352] | G: 8.9721 | D_A: 0.2760 | D_B: 0.3832 | Cycle: 0.5493 | Identity: 0.5110
Epoch [1/100] | Batch [50/352] | G: 10.1106 | D_A: 0.2620 | D_B: 0.4400 | Cycle: 0.6282 | Identity: 0.5738
Epoch [1/100] | Batch [60/352] | G: 8.5477 | D_A: 0.2858 | D_B: 0.2892 | Cycle: 0.5470 | Identity: 0.4807
Epoch [1/100] | Batch [70/352] | G: 9.4304 | D_A: 0.